# Anthropic extraction workflow

This template builds a small corpus and extracts structured records with Anthropic text and vision profiles. Set `ANTHROPIC_API_KEY` before starting Jupyter and choose a Messages API model available to your account.

In [ ]:
%env PS_DB=papers.db
%env PS_QUERY=lithium solid electrolyte
%env PS_RECIPE=sse
%env PS_MODEL=YOUR_ANTHROPIC_MODEL
%env PS_OUTPUT=temp_anthropic_materials.csv
%env PS_FINAL=anthropic_materials.csv

## Configure model profiles

Use separate identifiers if the chosen text model does not accept images. `ps_model_status` shows the effective provider, endpoint, and capabilities without printing the secret.

In [ ]:
%%bash
set -euo pipefail
test "$PS_MODEL" != "YOUR_ANTHROPIC_MODEL"
ps_model_config text --provider anthropic --model "$PS_MODEL"
ps_model_config vision --provider anthropic --model "$PS_MODEL"
ps_model_status

## Build the corpus

Search and download credentials are independent of the Anthropic key. This example uses OpenAlex discovery and every configured download source.

In [ ]:
%%bash
set -euo pipefail
ps_search "$PS_QUERY" "$PS_DB" --source openalex --count 25
ps_download "$PS_DB" --format both
ps_corpus_stats "$PS_DB"

## Extract and store records

Begin with five papers. Review the intermediate CSV before increasing the count or using `--force` for a deliberate rerun.

In [ ]:
%%bash
set -euo pipefail
ps_scrape "$PS_DB" "$PS_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PS_OUTPUT"
ps_store "$PS_DB" "$PS_OUTPUT" "$PS_FINAL" "$PS_RECIPE" --assume-yes
ps_status "$PS_DB"